In [1]:
import pandas as pd
import numpy as np
from decimal import Decimal, getcontext, InvalidOperation

PRO03_CSV_PATH = '../pro03.csv'

df = pd.read_csv(PRO03_CSV_PATH, encoding='cp949') #utf-8 or cp949

df

,연도,컨물동량,증감율,수출입소계,수입,수출,환적소계,환적점유율,환적증감율
0,2013,17686,3.75,8934,4424,4509,8748,49.47,7.38
1,2014,18683,5.64,9254,4596,4658,9429,50.47,7.78
2,2015,19469,4.20,9364,4714,4650,10105,51.91,7.17
3,2016,19456,-0.06,9620,4801,4819,9836,50.55,-2.67
4,2017,20493,5.33,10186,5042,5144,10225,49.90,3.96
5,2018,21663,5.70,10233,5117,5116,11429,52.76,11.77
6,2019,21992,1.52,10354,5192,5162,11638,52.92,1.83
7,2020,21824,-0.76,9804,4853,4951,12020,55.08,3.28
8,2021,22706,4.04,10433,5208,5225,12273,54.05,2.10
9,2022,22078,-2.77,10311,5133,5178,11766,53.29,-4.13


In [2]:
pd.DataFrame({'자료형': df.dtypes.astype('str'),
              '비결측수': df.notna().sum(),
              '결측수': df.isna().sum(),
              '결측률(%)': df.isna().mean()*100,
              '고유값 수': df.nunique(dropna=True)})

,자료형,비결측수,결측수,결측률(%),고유값 수
연도,int64,13,0,0.0,13
컨물동량,int64,13,0,0.0,13
증감율,float64,13,0,0.0,13
수출입소계,int64,13,0,0.0,13
수입,int64,13,0,0.0,13
수출,int64,13,0,0.0,13
환적소계,int64,13,0,0.0,13
환적점유율,float64,13,0,0.0,13
환적증감율,float64,13,0,0.0,13


In [3]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
연도,13.0,2019.000000,3.894440,2013.00,2016.00,2019.00,2022.00,2025.00
컨물동량,13.0,21422.153846,2169.439707,17686.00,19469.00,21824.00,22706.00,24882.00
증감율,13.0,2.991538,2.770773,-2.77,1.52,4.04,5.33,5.70
수출입소계,13.0,10071.230769,625.806966,8934.00,9620.00,10233.00,10433.00,10904.00
수입,13.0,5014.000000,310.466048,4424.00,4801.00,5117.00,5208.00,5409.00
수출,13.0,5057.076923,317.513638,4509.00,4819.00,5144.00,5225.00,5495.00
환적소계,13.0,11343.846154,1591.690864,8748.00,10105.00,11638.00,12273.00,14097.00
환적점유율,13.0,52.768462,2.233655,49.47,50.55,52.92,54.05,56.70
환적증감율,13.0,4.395385,4.459913,-4.13,2.10,4.40,7.38,11.77


In [4]:
col = df.columns.to_list
col

<bound method IndexOpsMixin.tolist of Index(['연도', ' 컨물동량', '증감율', '수출입소계', '수입', '수출', '환적소계', '환적점유율', '환적증감율'], dtype='str')>

In [5]:
data_cols = ['연도',' 컨물동량', '수입','수출','환적소계'] #원하는 수치만 선택

project = df[data_cols]

project

,연도,컨물동량,수입,수출,환적소계
0,2013,17686,4424,4509,8748
1,2014,18683,4596,4658,9429
2,2015,19469,4714,4650,10105
3,2016,19456,4801,4819,9836
4,2017,20493,5042,5144,10225
5,2018,21663,5117,5116,11429
6,2019,21992,5192,5162,11638
7,2020,21824,4853,4951,12020
8,2021,22706,5208,5225,12273
9,2022,22078,5133,5178,11766


In [6]:
# 연도를 인덱스로 설정
project = project.set_index('연도')

# 전년 대비 증감 수치
project['컨물동량증감'] = project[' 컨물동량'].diff()
project['수입증감'] = project['수입'].diff()
project['수출증감'] = project['수출'].diff()
project['환적소계증감'] = project['환적소계'].diff()

# 열 순서 정리
project = project[
    [
        '컨물동량증감',
        '수입', '수입증감',
        '수출', '수출증감',
        '환적소계','환적소계증감'
    ]
]

project

,컨물동량증감,수입,수입증감,수출,수출증감,환적소계,환적소계증감
연도,,,,,,,
2013,NaN,4424,NaN,4509,NaN,8748,NaN
2014,997.0,4596,172.0,4658,149.0,9429,681.0
2015,786.0,4714,118.0,4650,-8.0,10105,676.0
2016,-13.0,4801,87.0,4819,169.0,9836,-269.0
2017,1037.0,5042,241.0,5144,325.0,10225,389.0
2018,1170.0,5117,75.0,5116,-28.0,11429,1204.0
2019,329.0,5192,75.0,5162,46.0,11638,209.0
2020,-168.0,4853,-339.0,4951,-211.0,12020,382.0
2021,882.0,5208,355.0,5225,274.0,12273,253.0


In [7]:
# 수입증감률 + 수출증감률
rate_sum = project['수입증감'] + project['수출증감']

getcontext().prec = 28


def to_decimal(value):
    if pd.isna(value):
        return None

    value = str(value).strip().replace(',', '')

    if value in ['', '-', 'None', 'nan']:
        return None

    try:
        return Decimal(value)
    except InvalidOperation:
        return None
project['rate_sum'] = rate_sum

def calc_x(row):

    a = to_decimal(row['컨물동량증감'])
    b = to_decimal(row['환적소계증감'])
    rate = to_decimal(row['rate_sum'])

    if a is None or b is None or rate is None:
        return np.nan

    num = a - rate
    den = b - rate

    if den == 0:
        return np.nan

    return num / den


project['x'] = project.apply(calc_x, axis=1)

project

,컨물동량증감,수입,수입증감,수출,수출증감,환적소계,환적소계증감,rate_sum,x
연도,,,,,,,,,
2013,NaN,4424,NaN,4509,NaN,8748,NaN,NaN,NaN
2014,997.0,4596,172.0,4658,149.0,9429,681.0,321.0,1.877777777777777777777777778
2015,786.0,4714,118.0,4650,-8.0,10105,676.0,110.0,1.194346289752650176678445230
2016,-13.0,4801,87.0,4819,169.0,9836,-269.0,256.0,0.5123809523809523809523809524
2017,1037.0,5042,241.0,5144,325.0,10225,389.0,566.0,-2.661016949152542372881355932
2018,1170.0,5117,75.0,5116,-28.0,11429,1204.0,47.0,0.9706136560069144338807260156
2019,329.0,5192,75.0,5162,46.0,11638,209.0,121.0,2.363636363636363636363636364
2020,-168.0,4853,-339.0,4951,-211.0,12020,382.0,-550.0,0.4098712446351931330472103004
2021,882.0,5208,355.0,5225,274.0,12273,253.0,629.0,-0.6728723404255319148936170213
